# INSTRUCTOR SOLUTION: Cloud Deployment Cost Analysis and Scaling Decisions
## AIAT 125 — Unit 3: Cloud Deployment and Infrastructure

**⚠️ INSTRUCTOR USE ONLY — Do not distribute to students**

| Task | Points |
|---|---|
| Task 1: Monthly cost calculator | 25 |
| Task 2: Instance selector by SLA | 25 |
| Task 3: Horizontal scaling simulation | 25 |
| Task 4: Deployment recommendation report | 25 |

In [ ]:
import math
import numpy as np

# Cloud instance catalog (AWS pricing, us-east-1, on-demand, $/hour)
INSTANCE_CATALOG = {
    "t3.medium"   : 0.0416,
    "t3.large"    : 0.0832,
    "m5.xlarge"   : 0.192,
    "g4dn.xlarge" : 0.526,
    "g4dn.2xlarge": 0.752,
}

# Benchmark data for a 50MB sklearn model
INSTANCE_BENCHMARKS = {
    "t3.medium"   : {"max_rps": 20,   "p95_latency_ms": 450},
    "t3.large"    : {"max_rps": 50,   "p95_latency_ms": 190},
    "m5.xlarge"   : {"max_rps": 120,  "p95_latency_ms": 80},
    "g4dn.xlarge" : {"max_rps": 500,  "p95_latency_ms": 35},
    "g4dn.2xlarge": {"max_rps": 1000, "p95_latency_ms": 20},
}

print("Instance catalog:", list(INSTANCE_CATALOG.keys()))
print("Setup complete.")

---
## Task 1 — Monthly Cost Calculator (25 points)

In [ ]:
def calculate_monthly_cost(instance_type, hours_per_day=24, days_per_month=30):
    """
    Returns total monthly cost in USD for running an instance.
    Raises ValueError for invalid inputs.
    """
    # SOLUTION: Validate instance_type
    if instance_type not in INSTANCE_CATALOG:
        raise ValueError(f"Unknown instance type: '{instance_type}'. "
                         f"Valid types: {list(INSTANCE_CATALOG.keys())}")

    # SOLUTION: Validate hours_per_day
    if not (0 <= hours_per_day <= 24):
        raise ValueError(f"hours_per_day must be between 0 and 24, got {hours_per_day}")

    # SOLUTION: Validate days_per_month
    if not (1 <= days_per_month <= 31):
        raise ValueError(f"days_per_month must be between 1 and 31, got {days_per_month}")

    # SOLUTION: Compute and return cost
    price_per_hour = INSTANCE_CATALOG[instance_type]
    return price_per_hour * hours_per_day * days_per_month


print("Monthly cost (24h/day, 30 days):")
for instance in INSTANCE_CATALOG:
    cost = calculate_monthly_cost(instance, 24, 30)
    print(f"  {instance:15s}: ${cost:>8.2f}/month")

print("\nDev environment (8h/day, 22 working days):")
for instance in ["t3.medium", "t3.large"]:
    cost = calculate_monthly_cost(instance, 8, 22)
    print(f"  {instance:15s}: ${cost:>8.2f}/month")

# Validation
assert abs(calculate_monthly_cost("t3.medium", 24, 30) - (0.0416 * 24 * 30)) < 0.01
assert abs(calculate_monthly_cost("g4dn.xlarge", 8, 22) - (0.526 * 8 * 22)) < 0.01
try:
    calculate_monthly_cost("p4d.24xlarge")
    assert False, "Should raise ValueError"
except ValueError:
    pass
try:
    calculate_monthly_cost("t3.medium", hours_per_day=25)
    assert False, "Should raise ValueError"
except ValueError:
    pass
print("\nTask 1 PASSED")

---
## Task 2 — Instance Selector by SLA (25 points)

In [ ]:
def choose_instance(required_rps, latency_budget_ms):
    """
    Pick the cheapest instance meeting both RPS and latency SLA requirements.
    Returns a dict or None.
    """
    # SOLUTION: Filter candidates meeting both conditions
    candidates = [
        name for name, bench in INSTANCE_BENCHMARKS.items()
        if bench["max_rps"] >= required_rps
        and bench["p95_latency_ms"] <= latency_budget_ms
    ]

    if not candidates:
        return None

    # SOLUTION: Pick the one with the lowest cost_per_hour
    best = min(candidates, key=lambda name: INSTANCE_CATALOG[name])

    return {
        "instance": best,
        "cost_per_hour": INSTANCE_CATALOG[best],
        "max_rps": INSTANCE_BENCHMARKS[best]["max_rps"],
        "p95_latency_ms": INSTANCE_BENCHMARKS[best]["p95_latency_ms"],
    }


scenarios = [
    (10,  500, "Internal dashboard"),
    (40,  200, "Internal analytics API"),
    (80,  100, "Customer-facing API"),
    (400, 50,  "Real-time search feature"),
    (2000, 10, "High-frequency trading"),
]

print(f"{'Scenario':<28} {'RPS':>5} {'p95ms':>6} {'Chosen':>15} {'$/hr':>7}")
print("-" * 65)
for rps, lat, name in scenarios:
    result = choose_instance(rps, lat)
    if result:
        print(f"  {name:<26} {rps:>5} {lat:>6} "
              f"{result['instance']:>15} ${result['cost_per_hour']:>6.3f}")
    else:
        print(f"  {name:<26} {rps:>5} {lat:>6} {'No instance meets SLA':>22}")

# Validation
r1 = choose_instance(10, 500)
assert r1 is not None and r1["instance"] == "t3.medium", f"10rps/500ms → t3.medium, got {r1}"
r2 = choose_instance(80, 100)
assert r2 is not None and r2["instance"] == "m5.xlarge", f"80rps/100ms → m5.xlarge, got {r2}"
r3 = choose_instance(2000, 10)
assert r3 is None, "2000rps/10ms → None (no instance qualifies)"
print("\nTask 2 PASSED")

---
## Task 3 — Horizontal Scaling Simulation (25 points)

In [ ]:
def simulate_scaling(current_instance, target_rps):
    """
    Simulate horizontal scaling of current_instance to reach target_rps.
    Returns a dict with scaling details.
    """
    # SOLUTION: Get max_rps for current_instance
    max_rps_per_instance = INSTANCE_BENCHMARKS[current_instance]["max_rps"]

    # SOLUTION: Calculate replicas
    replicas = math.ceil(target_rps / max_rps_per_instance)

    # SOLUTION: Calculate total cost
    total_cost_per_hour = replicas * INSTANCE_CATALOG[current_instance]

    return {
        "strategy": "horizontal",
        "current_instance": current_instance,
        "replicas": replicas,
        "total_rps_capacity": replicas * max_rps_per_instance,
        "cost_per_hour": total_cost_per_hour,
    }


scaling_tests = [
    ("t3.large", 200,  "t3.large → 200 rps"),
    ("t3.large", 500,  "t3.large → 500 rps"),
    ("m5.xlarge", 500, "m5.xlarge → 500 rps"),
]

print(f"{'Scenario':<30} {'Replicas':>8} {'H-scale $/hr':>14} {'Best single $/hr':>18}")
print("-" * 74)
for instance, target, label in scaling_tests:
    h = simulate_scaling(instance, target)
    single = choose_instance(target, 9999)
    h_cost = h["cost_per_hour"]
    s_name = single["instance"] if single else "none"
    s_cost = single["cost_per_hour"] if single else float("inf")
    cheaper = "h-scale" if h_cost < s_cost else "single"
    print(f"  {label:<28} {h['replicas']:>8}     ${h_cost:>7.3f}         ${s_cost:>7.3f} ({s_name})  ← {cheaper} cheaper")

# Validation
r = simulate_scaling("t3.large", 200)
assert r is not None and r["strategy"] == "horizontal"
assert r["replicas"] == math.ceil(200 / INSTANCE_BENCHMARKS["t3.large"]["max_rps"])
assert r["total_rps_capacity"] >= 200
print("\nTask 3 PASSED")

---
## Task 4 — Deployment Recommendation Report (25 points)

In [ ]:
def deployment_recommendation(model_size_mb, expected_rps, latency_budget_ms, budget_per_month_usd):
    """
    Generate a deployment recommendation.
    Returns a dict with 6 required keys.
    """
    # SOLUTION: Use choose_instance to find the right instance
    result = choose_instance(expected_rps, latency_budget_ms)

    if result is None:
        notes = ("No available instance meets the SLA requirements "
                 f"({expected_rps} RPS, {latency_budget_ms}ms p95). "
                 "Consider relaxing the latency budget or using a multi-region setup.")
        if model_size_mb > 500:
            notes += " WARNING: Large model (>500MB) may require GPU instances."
        return {
            "recommended_instance": None,
            "monthly_cost_usd": 0.0,
            "within_budget": False,
            "expected_rps_capacity": 0,
            "p95_latency_ms": float("inf"),
            "notes": notes,
        }

    # SOLUTION: Compute monthly cost
    monthly_cost = calculate_monthly_cost(result["instance"], 24, 30)

    # SOLUTION: Check budget
    within_budget = monthly_cost <= budget_per_month_usd

    # SOLUTION: Build notes string
    notes = (f"Selected {result['instance']} — cheapest instance meeting "
             f"{expected_rps} RPS and {latency_budget_ms}ms p95 SLA. "
             f"Monthly cost ${monthly_cost:.2f} is "
             f"{'within' if within_budget else 'OVER'} the ${budget_per_month_usd} budget.")
    if model_size_mb > 500:
        notes += " WARNING: Large model (>500MB) may require GPU instances for adequate throughput."

    return {
        "recommended_instance": result["instance"],
        "monthly_cost_usd": round(monthly_cost, 2),
        "within_budget": within_budget,
        "expected_rps_capacity": result["max_rps"],
        "p95_latency_ms": result["p95_latency_ms"],
        "notes": notes,
    }


scenarios = [
    (50,   10,  500, 50,   "Dev chatbot prototype"),
    (200,  80,  100, 500,  "Production recommendation API"),
    (500,  400, 50,  2000, "Real-time image classification"),
    (2000, 5,   200, 30,   "Large LLM on tight budget"),
]

for model_mb, rps, latency, budget, name in scenarios:
    rec = deployment_recommendation(model_mb, rps, latency, budget)
    print(f"\n=== {name} ===")
    for k, v in rec.items():
        print(f"  {k}: {v}")

# Validation
rec = deployment_recommendation(50, 10, 500, 100)
assert isinstance(rec, dict)
for k in ["recommended_instance", "monthly_cost_usd", "within_budget",
          "expected_rps_capacity", "p95_latency_ms", "notes"]:
    assert k in rec, f"Missing key: {k}"
assert rec["recommended_instance"] == "t3.medium"
assert rec["within_budget"] == True
print("\nTask 4 PASSED")

In [ ]:
# --- Final summary ---
print("=" * 55)
print("UNIT 3 LAB — FINAL GATE")
print("=" * 55)
checks = {
    "Task 1: cost calculator validates inputs": True,
    "Task 2: instance selector picks cheapest valid": r1 is not None and r1["instance"] == "t3.medium",
    "Task 3: horizontal scaling returns correct replicas": r["replicas"] == math.ceil(200 / INSTANCE_BENCHMARKS["t3.large"]["max_rps"]),
    "Task 4: recommendation has all 6 keys": isinstance(rec, dict) and len(rec) >= 6,
}
for task, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)